In [ ]:
import pandas as pd
import yfinance as yf
import requests
import io
import warnings
warnings.filterwarnings('ignore')

### Question 1. [Index] S&P 500 Stocks Added to the Index

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
respond = requests.get(url, headers=headers)
respond.raise_for_status()

sp500 = pd.read_html(io.StringIO(respond.text))[0]
sp500["Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)

df = sp500[["Symbol", "Security", "Date added"]].copy()
df["Date added"] = pd.to_datetime(df["Date added"], errors="coerce")
df["Year added"] = df["Date added"].dt.year
df = df.rename(columns={"Symbol": "Ticker", "Security": "Name"})

yearly_summary = (
    df.groupby("Year added")
      .size()
      .reset_index(name="Stocks added each year")
)

# Highest number of additions starting from 2020
yearly_from_2020 = yearly_summary[yearly_summary["Year added"] >= 2020]
max_additions_row = yearly_from_2020.loc[yearly_from_2020["Stocks added each year"].idxmax()]
year_most_additions = int(max_additions_row["Year added"])
max_additions = int(max_additions_row["Stocks added each year"])

# Additional: stocks in the index for more than 20 years
cutoff_date = pd.Timestamp.today() - pd.DateOffset(years=20)
stocks_over_20_years = int((df["Date added"] < cutoff_date).sum())

print(df.head())
print("\nAdditions by year (from 2020):")
print(yearly_from_2020.to_string(index=False))
print(f"\nAnswer Q1: Year with most additions (from 2020) = {year_most_additions} ({max_additions} stocks)")
print(f"Additional: Stocks in index > 20 years = {stocks_over_20_years}")


  Ticker                 Name Date added  Year added
0    MMM                   3M 1957-03-04        1957
1    AOS          A. O. Smith 2017-07-26        2017
2    ABT  Abbott Laboratories 1957-03-04        1957
3   ABBV               AbbVie 2012-12-31        2012
4    ACN            Accenture 2011-07-06        2011

Additions by year (from 2020):
 Year added  Stocks added each year
       2020                      10
       2021                      10
       2022                      15
       2023                      15
       2024                      16
       2025                      18
       2026                      13

Answer Q1: Year with most additions (from 2020) = 2025 (18 stocks)
Additional: Stocks in index > 20 years = 224


### Question 2. [Macro] Indexes YTD (as of 21 August 2026)

In [3]:
indexes_ticker = [
    "^GSPC", "000001.SS", "^HSI", "^AXJO", "^NSEI",
    "^GSPTSE", "^GDAXI", "^FTSE", "^N225", "^MXX", "^BVSP"
]

indexes_df = yf.download(
    tickers=indexes_ticker,
    start="2026-01-01",
    end="2026-08-21",
    auto_adjust=True,
    progress=False,
)

close = indexes_df["Close"]

# YTD growth from each index's first/last available close
ytd = {}
for t in close.columns:
    s = close[t].dropna()
    if len(s) >= 2:
        ytd[t] = (s.iloc[-1] / s.iloc[0] - 1) * 100

ytd_returns = pd.Series(ytd).sort_values(ascending=False)

sp500_ytd = ytd_returns["^GSPC"]
better_than_sp500 = ytd_returns.drop(labels=["^GSPC"])
n_better = int((better_than_sp500 > sp500_ytd).sum())

print("YTD returns (%), 2026-01-01 to 2026-08-21:")
print(ytd_returns.round(2).to_string())
print(f"\nS&P 500 YTD = {sp500_ytd:.2f}%")
print(f"Answer Q2: {n_better} indexes (out of 10) beat the S&P 500")

YTD returns (%), 2026-01-01 to 2026-08-21:
^N225        27.75
^GSPTSE      14.06
^GSPC        11.41
^FTSE         8.01
^GDAXI        5.88
^BVSP         4.60
^AXJO         4.08
^MXX          0.32
^HSI         -2.43
000001.SS    -2.97
^NSEI        -7.32

S&P 500 YTD = 11.41%
Answer Q2: 2 indexes (out of 10) beat the S&P 500


### Question 3. [Index] S&P 500 Market Corrections Analysis

In [4]:
sp500_index = yf.download(
    tickers="^GSPC",
    start="1950-01-01",
    end="2026-08-21",
    auto_adjust=True,
    progress=False,
)

if isinstance(sp500_index.columns, pd.MultiIndex):
    sp500_index.columns = sp500_index.columns.get_level_values(0)

sp500_index = sp500_index[["Close"]].dropna().reset_index()
sp500_index.columns = ["Date", "Close"]
sp500_index["Date"] = pd.to_datetime(sp500_index["Date"])
sp500_index = sp500_index.sort_values("Date").reset_index(drop=True)
sp500_index

,Date,Close
0,1950-01-03,16.660000
1,1950-01-04,16.850000
2,1950-01-05,16.930000
3,1950-01-06,16.980000
4,1950-01-09,17.080000
...,...,...
19275,2026-08-14,7785.759766
19276,2026-08-17,7745.060059
19277,2026-08-18,7691.759766
19278,2026-08-19,7707.979980


In [5]:
sp500_index["Running Max"] = sp500_index["Close"].cummax()
ath_points = sp500_index[sp500_index["Close"] == sp500_index["Running Max"]].reset_index(drop=True)

drawdowns = []
for i in range(len(ath_points) - 1):
    peak_date = ath_points.loc[i, "Date"]
    peak_price = ath_points.loc[i, "Close"]
    next_ath_date = ath_points.loc[i + 1, "Date"]

    segment = sp500_index[
        (sp500_index["Date"] > peak_date) & (sp500_index["Date"] < next_ath_date)
    ]

    if segment.empty:
        continue

    min_idx = segment["Close"].idxmin()
    trough_date = segment.loc[min_idx, "Date"]
    trough_price = segment.loc[min_idx, "Close"]

    # (high - low) / high * 100
    drawdown_pct = (peak_price - trough_price) / peak_price * 100
    days = (trough_date - peak_date).days

    drawdowns.append({
        "Peak Date": peak_date,
        "Trough Date": trough_date,
        "Drawdown %": drawdown_pct,
        "Days": days,
    })

drawdowns_df = pd.DataFrame(drawdowns)

# Filter significant corrections: at least 5% drawdown
corrections = drawdowns_df[drawdowns_df["Drawdown %"] >= 5].copy()

duration_percentiles = corrections["Days"].quantile([0.25, 0.50, 0.75])
drawdown_percentiles = corrections["Drawdown %"].quantile([0.25, 0.50, 0.75])

print("Top 10 largest corrections by drawdown:")
print(
    corrections.nlargest(10, "Drawdown %")[
        ["Peak Date", "Trough Date", "Drawdown %", "Days"]
    ].to_string(index=False)
)
print("\nDuration percentiles (days):")
print(duration_percentiles.round(1).to_string())
print("\nDrawdown percentiles (%):")
print(drawdown_percentiles.round(2).to_string())
print(f"\nAnswer Q3: Median drawdown = {drawdown_percentiles.loc[0.50]:.2f}%")
print(f"Number of corrections >= 5%: {len(corrections)}")

Top 10 largest corrections by drawdown:
 Peak Date Trough Date  Drawdown %  Days
2007-10-09  2009-03-09   56.775388   517
2000-03-24  2002-10-09   49.146948   929
1973-01-11  1974-10-03   48.203593   630
1968-11-29  1970-05-26   36.061641   543
2020-02-19  2020-03-23   33.924960    33
1987-08-25  1987-12-04   33.509515   101
1961-12-12  1962-06-26   27.973568   196
1980-11-28  1982-08-12   27.113582   622
2022-01-03  2022-10-12   25.425097   282
1966-02-09  1966-10-07   22.177335   240

Duration percentiles (days):
0.25    22.0
0.50    40.5
0.75    86.2

Drawdown percentiles (%):
0.25     6.23
0.50     7.99
0.75    14.02

Answer Q3: Median drawdown = 7.99%
Number of corrections >= 5%: 74


### Question 4. [Stocks] Earnings Surprise Analysis for Amazon (AMZN)

In [ ]:
ticker = "AMZN"
ticker_obj = yf.Ticker(ticker)

earnings = ticker_obj.get_earnings_dates(limit=40)
earnings = earnings.dropna(subset=["Surprise(%)"]).reset_index()
earnings["Earnings Date"] = pd.to_datetime(earnings["Earnings Date"]).dt.tz_localize(None)

# Keep reported earnings from 2020-10-29 onward
earnings = earnings[earnings["Earnings Date"] >= "2020-10-29"].copy()

price_df = yf.download(
    tickers=ticker,
    start="2020-01-01",
    end="2026-08-21",
    auto_adjust=True,
    progress=False,
)
if isinstance(price_df.columns, pd.MultiIndex):
    price_df.columns = price_df.columns.get_level_values(0)

price_df = price_df[["Close"]].dropna().reset_index()
price_df["Date"] = pd.to_datetime(price_df["Date"]).dt.tz_localize(None)
price_df = price_df.sort_values("Date").reset_index(drop=True)

# Day 2 = earnings day; return = Close_Day3 / Close_Day1 - 1
rows = []
for _, erow in earnings.iterrows():
    edate = erow["Earnings Date"]
    day2_candidates = price_df.index[price_df["Date"] >= edate.normalize()]
    if len(day2_candidates) == 0:
        continue
    day2_idx = int(day2_candidates[0])
    if day2_idx == 0 or day2_idx + 1 >= len(price_df):
        continue

    day1_close = price_df.loc[day2_idx - 1, "Close"]
    day3_close = price_df.loc[day2_idx + 1, "Close"]
    ret_2d = day3_close / day1_close - 1

    rows.append({
        "Earnings Date": edate,
        "Surprise(%)": erow["Surprise(%)"],
        "Return_2d": ret_2d,
    })

earnings_returns = pd.DataFrame(rows)

positive = earnings_returns[earnings_returns["Surprise(%)"] > 0]
median_2d_return = positive["Return_2d"].median()
corr = earnings_returns[["Return_2d", "Surprise(%)"]].corr()

print(f"Earnings events used: {len(earnings_returns)}")
print(f"Positive surprises: {len(positive)}")
print(f"\nAnswer Q4: Median 2-day return after positive surprise = {median_2d_return * 100:.4f}%")
print("\nCorrelation (Return_2d vs Surprise(%)):")
print(corr.round(4).to_string())

Earnings events used: 24
Positive surprises: 20

Answer Q4: Median 2-day return after positive surprise = 0.3528%

Correlation (Return_2d vs Surprise(%)):
             Return_2d  Surprise(%)
Return_2d       1.0000       0.2217
Surprise(%)     0.2217       1.0000
